# One week of new behavioral-health organization NPIs

This notebook inspects the June 29–July 5, 2026 Practice Radar public release: **423 newly enumerated Type 2 organizations** selected from **2,229 screened** CMS NPPES records across eight disclosed behavioral-health taxonomy codes.

The state chart uses the aggregate edition receipt. The row-level file is intentionally a 15-record sample, so it is used for schema and quality checks—not population estimates.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

INPUT_ROOT = Path('/kaggle/input/weekly-behavioral-health-organization-npis')
SAMPLE_PATH = INPUT_ROOT / 'public' / 'sample.csv'
SUMMARY_PATH = INPUT_ROOT / 'public' / 'summary.json'

if SAMPLE_PATH.exists() and SUMMARY_PATH.exists():
    sample = pd.read_csv(SAMPLE_PATH, dtype={'npi': 'string', 'postal_code': 'string'})
    summary = json.loads(SUMMARY_PATH.read_text())
else:
    from urllib.request import urlopen
    raw_base = 'https://raw.githubusercontent.com/unitedideas/practice-radar-data/main/public'
    sample = pd.read_csv(f'{raw_base}/sample.csv', dtype={'npi': 'string', 'postal_code': 'string'})
    with urlopen(f'{raw_base}/summary.json') as response:
        summary = json.loads(response.read())

print(f"Edition: {summary['period']['start']} through {summary['period']['end']}")
print(f"Screened Type 2 organizations: {summary['newly_enumerated_type_2_organizations']:,}")
print(f"Selected records: {summary['selected_unique_records']:,}")
print(f"States and territories represented: {summary['states_and_territories']}")
print(f"Public sample rows: {len(sample)}")

## Where the selected records appeared

These are counts in one filtered weekly edition. They are **not** counts of active practices, clinicians, openings, or organizations ready to buy.

In [ ]:
state_counts = (
    pd.Series(summary['by_state'], name='selected_records')
      .sort_values(ascending=False)
      .head(12)
      .sort_values()
)

fig, ax = plt.subplots(figsize=(10, 6))
state_counts.plot.barh(ax=ax, color='#E6B84A')
ax.set_title('Top states in the June 29–July 5 filtered edition', loc='left', weight='bold')
ax.set_xlabel('Selected newly enumerated Type 2 organizations')
ax.set_ylabel('State')
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='x', alpha=0.2)
plt.tight_layout()
plt.show()

state_counts.sort_values(ascending=False).to_frame()

## Check the public sample before using it

The sample supports schema inspection, NPI matching tests, and workflow prototyping. The assertions below verify the published row count, NPI uniqueness, and required fields.

In [ ]:
required = ['npi', 'organization', 'city', 'state', 'enumeration_date', 'taxonomy_code']
assert len(sample) == 15, 'Expected the documented 15-row public sample'
assert sample['npi'].nunique() == len(sample), 'NPIs should be unique in the public sample'
assert sample[required].notna().all().all(), 'Required fields should be populated'

print('Quality checks passed.')
sample[['npi', 'organization', 'city', 'state', 'enumeration_date', 'taxonomy_code']].head(10)

## Responsible interpretation

An NPI enumeration is a fresh public registry event. It does not prove licensure, credentialing, active operation, independence, service availability, demand, budget, or buying intent. Verify every record before relying on it, and do not use this dataset for clinical or patient-care decisions.

- **Primary source:** [CMS NPPES data dissemination](https://download.cms.gov/nppes/NPI_Files.html)
- **Versioned public receipts and samples:** [Practice Radar data repository](https://github.com/unitedideas/practice-radar-data)
- **Method:** newly enumerated Type 2 organizations, deactivation removal, eight disclosed taxonomy matches, then organization/city/state deduplication